# Calgary Open Data Visualization Demo

## Import Libraries

In [1]:
from __future__ import annotations

from pathlib import Path

import geopandas as gpd
from bokeh.io import output_notebook
from bokeh.models import GeoJSONDataSource, HoverTool
from bokeh.palettes import Category20
from bokeh.plotting import figure, show

from calgary_dashboard.common.definitions import GEOMETRY_BUCKET_NAMES
from calgary_dashboard.common.io import list_subdirectories

output_notebook()

Loading BokehJS ...

## Load data

In [2]:
def resolve_data_dir() -> Path:
    """Prefer shared demos/data, with legacy fallbacks."""
    candidates = [
        Path("../data").resolve(),  # from demos/simple_subsection
        Path("./data").resolve(),  # legacy local data folder
        Path("demos/data").resolve(),  # when running from repo root
    ]
    for cand in candidates:
        if cand.is_dir():
            try:
                next(cand.iterdir())
                return cand
            except StopIteration:
                continue
    return candidates[0]


DATA_DIR = resolve_data_dir()
PLOT_CRS = "EPSG:4326"


def find_parquet_files(root: Path) -> list[Path]:
    """Find every *.parquet under each source/<snapshot>/features tree."""
    files: list[Path] = []
    for source_dir in list_subdirectories(root):
        for snapshot_dir in list_subdirectories(source_dir):
            features_dir = snapshot_dir / "features"
            if not features_dir.exists():
                continue
            for child in features_dir.iterdir():
                if child.is_dir() and child.name in GEOMETRY_BUCKET_NAMES:
                    files.extend(sorted(child.glob("*.parquet")))
                elif child.suffix == ".parquet":
                    files.append(child)
    return files


parquet_files = find_parquet_files(DATA_DIR)
for path in parquet_files:
    print(path.relative_to(DATA_DIR))

enmax/20260507/features/linestring/EnmaxServiceArea_features.parquet
enmax/20260507/features/linestring/SinglePhaseEstimatedRemainingLoadCapacityJuly2024_features.parquet
enmax/20260507/features/linestring/ThreePhaseEstimatedRemainingHostingCapacityFebruary2025_features.parquet
enmax/20260507/features/linestring/TwoAndThreePhaseEstimatedRemainingLoadCapacityDecember2025_features.parquet
enmax/20260507/features/linestring/TwoAndThreePhaseEstimatedRemainingLoadCapacityJuly2024_features.parquet
enmax/20260507/features/point/EnmaxDowntownProjects20230324_features.parquet
open_calgary/20260507/features/polygon/3wfx-q8xd_12FloodMap50ChanceOfOccurringInAnyYear_features.parquet
open_calgary/20260507/features/polygon/8rfi-nymv_150FloodMap2ChanceOfOccurringInAnyYear_features.parquet
open_calgary/20260507/features/polygon/w8wn-kuii_1100FloodMap1ChanceOfOccurringInAnyYear_features.parquet
open_calgary/20260507/features/multipolygon/g8ma-sywr_CityQuadrants_features.parquet
statcan/20260507/features

## Load every parquet as a GeoDataFrame

In [3]:
layers: dict[str, gpd.GeoDataFrame] = {}
for path in parquet_files:
    label = f"{path.parents[3].name}/{path.stem.replace('_features', '')}"
    try:
        gdf = gpd.read_parquet(path)
        if gdf.crs is not None:
            gdf = gdf.to_crs(PLOT_CRS)
        layers[label] = gdf
    except Exception as exc:
        print(f"skip {label}: {exc}")

for label, gdf in layers.items():
    print(f"{label:70s} rows={len(gdf):>6}  geom={gdf.geom_type.iloc[0] if len(gdf) else 'n/a'}  crs={gdf.crs}")

enmax/EnmaxServiceArea                                                 rows=     1  geom=LineString  crs=EPSG:4326
enmax/SinglePhaseEstimatedRemainingLoadCapacityJuly2024                rows= 15434  geom=LineString  crs=EPSG:4326
enmax/ThreePhaseEstimatedRemainingHostingCapacityFebruary2025          rows=  5141  geom=LineString  crs=EPSG:4326
enmax/TwoAndThreePhaseEstimatedRemainingLoadCapacityDecember2025       rows= 17125  geom=LineString  crs=EPSG:4326
enmax/TwoAndThreePhaseEstimatedRemainingLoadCapacityJuly2024           rows= 25289  geom=LineString  crs=EPSG:4326
enmax/EnmaxDowntownProjects20230324                                    rows=     5  geom=Point  crs=EPSG:4326
open_calgary/3wfx-q8xd_12FloodMap50ChanceOfOccurringInAnyYear          rows=    40  geom=Polygon  crs={"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic S

In [4]:
# Example: first loaded layer
first_key = next(iter(layers))
first_key, layers[first_key].head(2)


('enmax/EnmaxServiceArea',
                                             geometry  OBJECTID  Id  ORIG_FID  \
 0  LINESTRING (-114.00103 0.00046, -114.00102 0.0...         1   0         1   
 
    Shape__Length  
 0   163326.48142  )

## Downtown 1 mi × 1 mi subset

Clip each layer to a **1 statute mile × 1 statute mile** axis-aligned box centred on downtown Calgary (~City Hall / Stephen Avenue). Geometry is measured in **UTM zone 11N (EPSG:32611)** metres, then clipped in **WGS84** to match `layers`. This keeps the Bokeh cell fast for large line datasets.

In [5]:
from shapely.geometry import box

# Approximate downtown Calgary centre (WGS84). ~City Hall / +15 / Stephen Avenue corridor.
DOWNTOWN_LON = -114.065
DOWNTOWN_LAT = 51.045
MILES = 1.0
HALF_MILES = MILES / 2.0  # half-width from centre → 1 mi total extent
METERS_PER_MILE = 1609.344
_CLIP_WORK = "EPSG:32611"  # UTM 11N for metre-accurate box

half_m = HALF_MILES * METERS_PER_MILE
center_pt = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([DOWNTOWN_LON], [DOWNTOWN_LAT]),
    crs=PLOT_CRS,
).to_crs(_CLIP_WORK)
cx, cy = float(center_pt.geometry.iloc[0].x), float(center_pt.geometry.iloc[0].y)
downtown_box = box(cx - half_m, cy - half_m, cx + half_m, cy + half_m)
downtown_mask = gpd.GeoDataFrame(geometry=[downtown_box], crs=_CLIP_WORK).to_crs(PLOT_CRS)

bounds_ll = downtown_mask.total_bounds
print(
    "Clip bbox (WGS84):",
    f"lon [{bounds_ll[0]:.5f}, {bounds_ll[2]:.5f}]  lat [{bounds_ll[1]:.5f}, {bounds_ll[3]:.5f}]",
)

layers_full = dict(layers)
layers = {}
for label, gdf in layers_full.items():
    if gdf is None or gdf.empty:
        continue
    g = gdf
    if g.crs is None:
        g = g.set_crs(PLOT_CRS)
    elif g.crs != PLOT_CRS:
        g = g.to_crs(PLOT_CRS)
    try:
        clipped = g.clip(downtown_mask)
    except Exception as exc:
        print(f"clip skip {label}: {exc}")
        continue
    if clipped.empty:
        print(f"clip empty (no overlap): {label}")
        continue
    layers[label] = clipped

print("--- row counts after clip ---")
for label, gdf in layers.items():
    n0 = len(layers_full[label])
    print(f"{label:60s}  {n0:>7} -> {len(gdf):>7}")


Clip bbox (WGS84): lon [-114.07692, -114.05308]  lat [51.03748, 51.05251]
clip empty (no overlap): enmax/EnmaxServiceArea
clip empty (no overlap): enmax/SinglePhaseEstimatedRemainingLoadCapacityJuly2024
clip empty (no overlap): enmax/ThreePhaseEstimatedRemainingHostingCapacityFebruary2025
clip empty (no overlap): enmax/TwoAndThreePhaseEstimatedRemainingLoadCapacityDecember2025
clip empty (no overlap): enmax/TwoAndThreePhaseEstimatedRemainingLoadCapacityJuly2024
clip empty (no overlap): enmax/EnmaxDowntownProjects20230324
--- row counts after clip ---
open_calgary/3wfx-q8xd_12FloodMap50ChanceOfOccurringInAnyYear       40 ->       1
open_calgary/8rfi-nymv_150FloodMap2ChanceOfOccurringInAnyYear     1448 ->      16
open_calgary/w8wn-kuii_1100FloodMap1ChanceOfOccurringInAnyYear     1221 ->       5
open_calgary/g8ma-sywr_CityQuadrants                                4 ->       3
data/98100015_PopulationAndDwellingCountsCanada                  2059 ->      30


## Interactive map (Bokeh)

In [6]:
import html

p = figure(
    width=950,
    height=900,
    title="Calgary downtown — 1 mi × 1 mi clip (interactive)",
    match_aspect=True,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    x_axis_label="Longitude",
    y_axis_label="Latitude",
)

palette = Category20[20]

for idx, (label, gdf) in enumerate(layers.items()):
    if gdf is None or gdf.empty:
        continue
    short_label = label if len(label) <= 56 else f"{label[:53]}..."
    color = palette[idx % len(palette)]
    geom0 = gdf.geometry.iloc[0]
    geom_kind = geom0.geom_type if geom0 is not None else ""

    try:
        geojson = gdf.to_json()
    except Exception as exc:
        print(f"skip geojson {short_label}: {exc}")
        continue

    src = GeoJSONDataSource(geojson=geojson)

    if geom_kind in ("Polygon", "MultiPolygon"):
        r = p.patches(
            xs="xs",
            ys="ys",
            source=src,
            fill_color=color,
            fill_alpha=0.35,
            line_color=color,
            line_width=0.6,
            legend_label=short_label,
        )
    elif geom_kind in ("LineString", "MultiLineString", "LinearRing"):
        r = p.multi_line(
            xs="xs",
            ys="ys",
            source=src,
            line_color=color,
            line_width=1.6,
            legend_label=short_label,
        )
    else:
        r = p.scatter(
            x="x",
            y="y",
            source=src,
            size=7,
            color=color,
            alpha=0.85,
            legend_label=short_label,
        )

    tip = html.escape(label)
    p.add_tools(
        HoverTool(
            renderers=[r],
            tooltips=[("layer", tip), ("(lon, lat)", "($x{0.000}, $y{0.000})")],
        )
    )

p.legend.click_policy = "hide"
p.legend.label_text_font_size = "9pt"
p.legend.spacing = 2

show(p)
